# Week 4 — Lab 3: Retrieval-Augmented Generation (RAG)

In Lab 1, you saw that LLMs have a knowledge cutoff and can hallucinate confidently when they don't actually know something. RAG (Retrieval-Augmented Generation) is the standard technique used to fix this: instead of relying only on what the model memorized during training, you give it relevant, up-to-date information at the moment you ask your question, and the model generates its answer grounded in that information.

This lab builds a complete, working RAG pipeline from scratch using free, open-source tools — no API key or paid account required.

**After this lab you will be able to:**
- Explain why RAG exists and what problem it solves
- Build a simple knowledge base and generate embeddings for it
- Retrieve the most relevant pieces of information for a given query
- Combine retrieved context with a query to generate a grounded answer
- Compare model behavior with and without retrieval

**Instructions:**
- Write your code between the `### YOUR CODE HERE ###` and `### END ###` markers.
- Run each cell with **Shift + Enter**.

## Setup

We'll use `sentence-transformers` for generating embeddings and `transformers` for text generation.

In [1]:
!pip install -q sentence-transformers transformers torch scikit-learn

## Why RAG?

Recall from Lab 1: an LLM only knows what was in its training data, up to its knowledge cutoff. It cannot know about your company's internal documents, a product launched last week, or anything specific that wasn't part of its training. If you ask it anyway, it may hallucinate a plausible-sounding but wrong answer instead of saying "I don't know."

RAG solves this with two steps combined:
1. **Retrieval**: given a question, search a knowledge base (documents, database, wiki, etc.) for the most relevant pieces of text.
2. **Generation**: feed those retrieved pieces of text to the LLM along with the original question, so it can generate an answer grounded in real, provided information instead of relying purely on memorized patterns.

We'll build a tiny example of this end-to-end below.

## Step 1: Build a Knowledge Base

Below is a small knowledge base about a fictional company, NovaTech Inc. No language model has ever seen this information during training, since we just made it up — this simulates exactly the situation RAG is designed for: answering questions about information the model could not possibly know on its own.

In [2]:
knowledge_base = [
    "NovaTech Inc. was founded in 2019 and is headquartered in Austin, Texas. The company specializes in consumer audio electronics.",
    "NovaTech's flagship product is the Aurora X wireless headphones, released in March 2024, featuring 40-hour battery life and active noise cancellation.",
    "NovaTech offers a 30-day return policy on all products. Items must be returned in original packaging with proof of purchase for a full refund.",
    "The standard warranty on NovaTech products is 12 months from the date of purchase, covering manufacturing defects but not accidental damage.",
    "NovaTech customer support is available Monday to Friday, 9 AM to 6 PM Central Time, via email and live chat on their website.",
    "NovaTech offers three pricing tiers for the Aurora X: Standard ($199), Pro ($249) with extra ear tip sizes, and Limited Edition ($299) with a premium carrying case.",
    "In 2025, NovaTech launched the Aurora X2, an upgraded version with improved noise cancellation and a companion mobile app for custom sound profiles.",
    "NovaTech does not offer international shipping outside the United States and Canada as of this year.",
]

print(f"Knowledge base has {len(knowledge_base)} chunks.")

Knowledge base has 8 chunks.


## Step 2: Generate Embeddings

An embedding is a numeric vector representation of text, where texts with similar meaning end up close together in vector space. We'll use a small, free sentence embedding model to convert each chunk of our knowledge base into a vector.

In [3]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('all-MiniLM-L6-v2')

kb_embeddings = embedder.encode(knowledge_base)
print("Shape of embeddings:", kb_embeddings.shape)
print("Each chunk is now a vector of", kb_embeddings.shape[1], "numbers.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Shape of embeddings: (8, 384)
Each chunk is now a vector of 384 numbers.


## Step 3: Retrieval — Finding Relevant Chunks

To answer a question, we embed the question the same way, then use cosine similarity to find which knowledge base chunks are closest in meaning to the question.

In [4]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def retrieve(query, top_k=2):
    query_embedding = embedder.encode([query])
    similarities = cosine_similarity(query_embedding, kb_embeddings)[0]
    top_indices = np.argsort(similarities)[::-1][:top_k]
    return [(knowledge_base[i], similarities[i]) for i in top_indices]

query = "What is NovaTech's return policy?"
results = retrieve(query, top_k=2)

for chunk, score in results:
    print(f"Score: {score:.3f} | {chunk}")

Score: 0.800 | NovaTech offers a 30-day return policy on all products. Items must be returned in original packaging with proof of purchase for a full refund.
Score: 0.570 | The standard warranty on NovaTech products is 12 months from the date of purchase, covering manufacturing defects but not accidental damage.


## Practice: Testing Retrieval

**Exercise:** Try the `retrieve()` function with three different queries of your own about NovaTech (e.g., about pricing, warranty, or the Aurora X2). For each, print the top 2 retrieved chunks and their similarity scores. Note whether the retrieval found the correct information each time.

In [5]:
### YOUR CODE HERE ###
test_queries = [
    "How much does the Aurora X Pro cost?",
    "How long is the warranty?",
    "What is new in the Aurora X2?",
]

for q in test_queries:
    print("Query:", q)
    for chunk, score in retrieve(q, top_k=2):
        print(f"  Score: {score:.3f} | {chunk}")
    print()

# Observation: for each query the top-1 chunk is the correct one
# (pricing tiers, warranty, Aurora X2 launch). The 2nd chunk is
# usually less relevant, with a noticeably lower score.
### END ###

Query: How much does the Aurora X Pro cost?
  Score: 0.734 | NovaTech offers three pricing tiers for the Aurora X: Standard ($199), Pro ($249) with extra ear tip sizes, and Limited Edition ($299) with a premium carrying case.
  Score: 0.558 | In 2025, NovaTech launched the Aurora X2, an upgraded version with improved noise cancellation and a companion mobile app for custom sound profiles.

Query: How long is the warranty?
  Score: 0.513 | The standard warranty on NovaTech products is 12 months from the date of purchase, covering manufacturing defects but not accidental damage.
  Score: 0.347 | NovaTech offers a 30-day return policy on all products. Items must be returned in original packaging with proof of purchase for a full refund.

Query: What is new in the Aurora X2?
  Score: 0.666 | In 2025, NovaTech launched the Aurora X2, an upgraded version with improved noise cancellation and a companion mobile app for custom sound profiles.
  Score: 0.569 | NovaTech's flagship product is the 

## Step 4: Loading a Generation Model

Now we need a model that can read retrieved context and generate an answer. We'll use `flan-t5-base`, a free, instruction-tuned model well-suited to answering questions given context.

In [6]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

gen_tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
gen_model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")

def ask_model(prompt, max_new_tokens=60):
    input_ids = gen_tokenizer(prompt, return_tensors="pt").input_ids
    outputs = gen_model.generate(input_ids, max_new_tokens=max_new_tokens)
    return gen_tokenizer.decode(outputs[0], skip_special_tokens=True)

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

## Step 5: Asking WITHOUT Retrieval

Let's first ask the model directly, with no retrieved context at all — just the raw question. Since this information was never in the model's training data (we invented NovaTech ourselves), watch what happens.

In [7]:
question = "What is NovaTech's return policy?"

answer_no_rag = ask_model(question)
print("Question:", question)
print("Answer (no RAG):", answer_no_rag)

Question: What is NovaTech's return policy?
Answer (no RAG): a one-year warranty


The model likely gives a vague, generic, or made-up answer — because it genuinely has no information about NovaTech. This is the exact failure mode RAG is designed to fix.

## Step 6: Asking WITH Retrieval (Full RAG Pipeline)

Now let's combine retrieval and generation: retrieve the most relevant chunks, then include them in the prompt so the model has to ground its answer in that information.

In [8]:
def rag_answer(question, top_k=2):
    retrieved = retrieve(question, top_k=top_k)
    context = "\n".join([chunk for chunk, score in retrieved])

    prompt = f"""Answer the question using only the context below. If the context doesn't contain the answer, say "I don't have that information."

Context:
{context}

Question: {question}
Answer:"""

    return ask_model(prompt, max_new_tokens=60), context

question = "What is NovaTech's return policy?"
answer_with_rag, used_context = rag_answer(question)

print("Question:", question)
print("\nRetrieved context:")
print(used_context)
print("\nAnswer (with RAG):", answer_with_rag)

Question: What is NovaTech's return policy?

Retrieved context:
NovaTech offers a 30-day return policy on all products. Items must be returned in original packaging with proof of purchase for a full refund.
The standard warranty on NovaTech products is 12 months from the date of purchase, covering manufacturing defects but not accidental damage.

Answer (with RAG): 30-day return policy


Compare this answer to the one from Step 5. The RAG version should be accurate and grounded in the actual NovaTech policy, because the model was given the real information instead of having to guess.

## Practice: With vs. Without RAG

**Exercise:** Pick two more questions about NovaTech (e.g., about the Aurora X2 or pricing tiers). For each, get the answer both with and without RAG, and print both side by side. Note which answer is more accurate for each question.

In [9]:
### YOUR CODE HERE ###
questions = [
    "What new features does the Aurora X2 have?",
    "How much is the Limited Edition Aurora X?",
]

for q in questions:
    no_rag = ask_model(q)
    with_rag, ctx = rag_answer(q)
    print("Question:", q)
    print("  Without RAG:", no_rag)
    print("  With RAG   :", with_rag)
    print("-" * 70)

# Observation: without RAG the model guesses or gives something generic;
# with RAG it answers from the retrieved NovaTech chunks
# (improved noise cancellation + companion app; $299).
### END ###

Question: What new features does the Aurora X2 have?
  Without RAG: a new X2 engine
  With RAG   : improved noise cancellation and a companion mobile app for custom sound profiles
----------------------------------------------------------------------
Question: How much is the Limited Edition Aurora X?
  Without RAG: $29,900
  With RAG   : $299
----------------------------------------------------------------------


## Practice: When Retrieval Fails

RAG isn't perfect — if the knowledge base doesn't actually contain the answer, or if the question is worded very differently from the source text, retrieval can pull irrelevant chunks, and the model may still produce a poor answer.

**Exercise:** Ask a question that has nothing to do with anything in the knowledge base (e.g., "What programming language does NovaTech use for its app?" — not covered in our knowledge base). Run it through the full RAG pipeline and observe what happens. Does the model correctly say it doesn't know, or does it still attempt an answer?

In [10]:
### YOUR CODE HERE ###
question = "What programming language does NovaTech use for its app?"

print("Retrieved chunks:")
for chunk, score in retrieve(question, top_k=2):
    print(f"  Score: {score:.3f} | {chunk}")

answer, ctx = rag_answer(question)
print("\nQuestion:", question)
print("Answer (with RAG):", answer)

# Observation: retrieval still returns the "closest" chunks (e.g. the Aurora X2
# / mobile app chunk) even though none contain the answer. A small model like
# flan-t5-base may say "I don't have that information." or may guess/copy
# something from the context -- compare with what you see when you run it.
### END ###

Retrieved chunks:
  Score: 0.462 | NovaTech Inc. was founded in 2019 and is headquartered in Austin, Texas. The company specializes in consumer audio electronics.
  Score: 0.441 | NovaTech customer support is available Monday to Friday, 9 AM to 6 PM Central Time, via email and live chat on their website.

Question: What programming language does NovaTech use for its app?
Answer (with RAG): I don't have that information


**What to remember from this lab:**
- RAG combines retrieval (finding relevant information) with generation (producing an answer) to ground LLM responses in real, provided data
- Embeddings let us measure semantic similarity between a query and a knowledge base, not just exact keyword matches
- Without RAG, a model has no way to know information outside its training data and will often produce vague or incorrect answers instead of admitting it doesn't know
- RAG quality depends heavily on the quality of retrieval — if the right information isn't retrieved, the generated answer will still be wrong

## Lab Tasks

Complete the following tasks in this notebook, below this cell.

1. **Build Your Own Knowledge Base:** Create a new knowledge base of 6-8 text chunks about a topic of your choice (a fictional product, a hobby, a made-up historical event — anything not likely to be well-known training data). Generate embeddings for it.
2. **Implement Retrieval:** Using your new knowledge base, write and test a retrieval function (you can reuse the `retrieve()` pattern above) with at least 3 different queries. Print the retrieved chunks and similarity scores for each.
3. **Full RAG Comparison:** For at least 2 questions about your knowledge base, generate answers both with and without RAG. Present both answers clearly and note the difference in accuracy.
4. **Retrieval Failure Test:** Ask one question that is NOT covered by your knowledge base and run it through your RAG pipeline. Report what happened and whether the model handled the "I don't know" case well.
5. **Reflection:** In a markdown cell, explain in your own words: (a) why RAG is useful for real-world applications, and (b) one limitation of the RAG approach you observed while doing this lab.

In [11]:
# Task 1: Build Your Own Knowledge Base
# Topic: a fictional smart lamp from a made-up company, "Lumora" by Brightwell Labs
my_kb = [
    "Brightwell Labs was founded in 2021 in Portland, Oregon, and builds smart home lighting products.",
    "The Lumora Smart Lamp, released in September 2023, can produce 16 million colors and be controlled with a mobile app or voice commands.",
    "Lumora's Sunrise Mode gradually brightens the lamp over 30 minutes before the user's alarm time to help them wake up naturally.",
    "The Lumora Smart Lamp comes in two sizes: the Lumora Mini (desk size) for $59 and the Lumora Floor (standing) for $129.",
    "Brightwell Labs offers a 2-year warranty on all Lumora lamps, but the warranty does not cover damage from water exposure.",
    "Lumora lamps connect via Wi-Fi 2.4 GHz and Bluetooth 5.0, and they work with Alexa and Google Assistant but not Apple HomeKit.",
    "The Lumora Focus Mode uses cool white light at 6500K to help with studying, while Relax Mode uses warm 2700K light.",
    "Brightwell Labs ships Lumora lamps to the United States, United Kingdom, and Australia, with free shipping on orders over $100.",
]

my_embeddings = embedder.encode(my_kb)
print(f"My knowledge base has {len(my_kb)} chunks.")
print("Embedding shape:", my_embeddings.shape)

My knowledge base has 8 chunks.
Embedding shape: (8, 384)


In [12]:
# Task 2: Implement Retrieval
def my_retrieve(query, top_k=2):
    q_emb = embedder.encode([query])
    sims = cosine_similarity(q_emb, my_embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [(my_kb[i], sims[i]) for i in top_idx]

task2_queries = [
    "How much does the Lumora Mini cost?",
    "Which smart assistants does the lamp work with?",
    "How does the lamp help me wake up?",
    "Where does Brightwell Labs ship?",
]

for q in task2_queries:
    print("Query:", q)
    for chunk, score in my_retrieve(q, top_k=2):
        print(f"  Score: {score:.3f} | {chunk}")
    print()

Query: How much does the Lumora Mini cost?
  Score: 0.577 | The Lumora Smart Lamp comes in two sizes: the Lumora Mini (desk size) for $59 and the Lumora Floor (standing) for $129.
  Score: 0.470 | Brightwell Labs ships Lumora lamps to the United States, United Kingdom, and Australia, with free shipping on orders over $100.

Query: Which smart assistants does the lamp work with?
  Score: 0.521 | Lumora lamps connect via Wi-Fi 2.4 GHz and Bluetooth 5.0, and they work with Alexa and Google Assistant but not Apple HomeKit.
  Score: 0.519 | The Lumora Smart Lamp, released in September 2023, can produce 16 million colors and be controlled with a mobile app or voice commands.

Query: How does the lamp help me wake up?
  Score: 0.626 | Lumora's Sunrise Mode gradually brightens the lamp over 30 minutes before the user's alarm time to help them wake up naturally.
  Score: 0.301 | The Lumora Smart Lamp comes in two sizes: the Lumora Mini (desk size) for $59 and the Lumora Floor (standing) for $12

In [13]:
# Task 3: Full RAG Comparison
def my_rag_answer(question, top_k=2):
    retrieved = my_retrieve(question, top_k=top_k)
    context = "\n".join([chunk for chunk, score in retrieved])
    prompt = f"""Answer the question using only the context below. If the context doesn't contain the answer, say "I don't have that information."

Context:
{context}

Question: {question}
Answer:"""
    return ask_model(prompt, max_new_tokens=60), context

task3_questions = [
    "How long is the warranty on Lumora lamps?",
    "How much does the Lumora Floor lamp cost?",
]

for q in task3_questions:
    no_rag = ask_model(q)
    with_rag, ctx = my_rag_answer(q)
    print("Question   :", q)
    print("Without RAG:", no_rag)
    print("With RAG   :", with_rag)
    print("Ground truth: warranty = 2 years; Lumora Floor = $129")
    print("-" * 70)

Question   : How long is the warranty on Lumora lamps?
Without RAG: 365 days
With RAG   : 2 years
Ground truth: warranty = 2 years; Lumora Floor = $129
----------------------------------------------------------------------
Question   : How much does the Lumora Floor lamp cost?
Without RAG: $29
With RAG   : $129.
Ground truth: warranty = 2 years; Lumora Floor = $129
----------------------------------------------------------------------


In [14]:
# Task 4: Retrieval Failure Test
# Not covered anywhere in my knowledge base:
q = "What is the battery capacity of the Lumora lamp?"

print("Retrieved chunks:")
for chunk, score in my_retrieve(q, top_k=2):
    print(f"  Score: {score:.3f} | {chunk}")

answer, ctx = my_rag_answer(q)
print("\nQuestion:", q)
print("Answer (with RAG):", answer)
print("\nCorrect behaviour would be: 'I don't have that information.'")

Retrieved chunks:
  Score: 0.514 | The Lumora Smart Lamp, released in September 2023, can produce 16 million colors and be controlled with a mobile app or voice commands.
  Score: 0.472 | The Lumora Smart Lamp comes in two sizes: the Lumora Mini (desk size) for $59 and the Lumora Floor (standing) for $129.

Question: What is the battery capacity of the Lumora lamp?
Answer (with RAG): I don't have that information.

Correct behaviour would be: 'I don't have that information.'


## Task 5: Reflection

**(a) Why RAG is useful in real-world applications.**
An LLM only knows what was in its training data, so it can't answer questions about private, new, or niche information (a company's policies, a product launched last week, internal documents), and when asked anyway it tends to guess confidently. RAG fixes this by searching a knowledge base at question time and handing the relevant text to the model, so answers are grounded in real, current information. The knowledge base can be updated at any time without retraining the model, and the retrieved chunks can be shown to the user as sources they can verify. This makes RAG practical for customer-support bots, internal company assistants, and documentation search.

**(b) A limitation of RAG.**
RAG is only as good as its retrieval and the model reading the context. In the retrieval-failure test, the retriever *always* returns the top-k "closest" chunks, even when none of them contain the answer (e.g. a battery-capacity question still pulls up lamp-related chunks). A small model like `flan-t5-base` can then either guess from loosely related context or fail to follow the "say you don't know" instruction reliably, producing a wrong or misleading answer that looks grounded. Fixes include a minimum similarity-score threshold, better chunking, a larger model, and re-ranking.